# Notebook 06: Model Evaluation
## Forecasting Education Performance -- Tanzania BEST Datasets (2020-2024)
**Author:** Habil Masawika | **Project:** Tanzania BEST ML Forecasting

---

### Objectives
1. Conduct thorough diagnostic evaluation of the best model on the 2024 test set
2. Analyse residuals for bias, heteroscedasticity, and normality
3. Examine fold-by-fold cross-validation stability
4. Generate learning curves to assess bias-variance trade-off
5. Produce prediction error breakdown by year and region
6. Compare multiple cross-validation strategies (KFold, RepeatedKFold, LOGO, TimeSeriesSplit)

In [ ]:
import sys, warnings
sys.path.insert(0, '/home/claude/BEST-ML-Forecasting/src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import LeaveOneGroupOut, KFold, RepeatedKFold, TimeSeriesSplit, cross_val_score

from utilities import setup_logging, set_seeds, ProjectPaths, load_model, save_dataframe
from feature_engineering import get_model_features
from preprocessing import scale_features
from models import compute_metrics
from evaluation import (actual_vs_predicted_plot, residual_diagnostics_plot,
                         learning_curve_plot, model_comparison_bar_chart,
                         error_distribution_by_year)

set_seeds(42)
logger = setup_logging()
paths  = ProjectPaths()

panel_fe = pd.read_csv(paths.processed('best_panel_features.csv'))
model_feats = list(pd.read_csv(paths.table('model_features.csv'))['feature'])
TARGET = 'csee_pass_rate'

model_df = panel_fe[['year','region'] + model_feats + [TARGET]].dropna(
    subset=model_feats + [TARGET])

TRAIN_YEARS = [2020, 2021, 2022, 2023]
train = model_df[model_df['year'].isin(TRAIN_YEARS)]
test  = model_df[model_df['year'] == 2024]

X_train_raw = train[model_feats].values
y_train     = train[TARGET].values
X_test_raw  = test[model_feats].values
y_test      = test[TARGET].values

scaler      = load_model(paths.model('feature_scaler.pkl'))
X_train_sc  = scaler.transform(X_train_raw)
X_test_sc   = scaler.transform(X_test_raw)
groups      = train['year'].values

# Load best model
best_model = load_model(paths.model('gradient_boosting.pkl'))
best_name  = 'Gradient Boosting'
print(f"Loaded model: {best_name}")
print(f"Train: {X_train_sc.shape} | Test: {X_test_sc.shape}")

In [ ]:
# ── Full metrics on test set ──────────────────────────────────────────────
best_model.fit(X_train_sc, y_train)
y_pred = best_model.predict(X_test_sc)

metrics = compute_metrics(y_test, y_pred, n_features=len(model_feats))
print(f"\nTest Set Metrics -- {best_name}:")
for k, v in metrics.items():
    print(f"  {k:<15} {v}")

In [ ]:
# ── Actual vs predicted ───────────────────────────────────────────────────
fig = actual_vs_predicted_plot(y_test, y_pred, best_name,
      save_path=paths.fig('nb06_actual_vs_predicted.png'))
plt.show()

In [ ]:
# ── Residual diagnostics ─────────────────────────────────────────────────
fig = residual_diagnostics_plot(y_test, y_pred, best_name,
      save_path=paths.fig('nb06_residual_diagnostics.png'))
plt.show()
residuals = y_test - y_pred
print(f"Residual stats:")
print(f"  Mean:   {residuals.mean():.4f}")
print(f"  Std:    {residuals.std():.4f}")
print(f"  Max abs:{np.abs(residuals).max():.4f}")

In [ ]:
# ── Cross-validation across multiple strategies ───────────────────────────
strategies = {
    'KFold (k=5)':        KFold(n_splits=5, shuffle=True, random_state=42),
    'RepeatedKFold':      RepeatedKFold(n_splits=5, n_repeats=3, random_state=42),
    'Leave-One-Year-Out': list(LeaveOneGroupOut().split(X_train_sc, y_train, groups)),
    'TimeSeriesSplit':    TimeSeriesSplit(n_splits=4),
}

from sklearn.ensemble import GradientBoostingRegressor
gb_fresh = GradientBoostingRegressor(**best_model.get_params())

cv_strategy_results = []
for name, cv in strategies.items():
    g = groups if 'Year' in name else None
    scores_mae = cross_val_score(gb_fresh, X_train_sc, y_train, cv=cv,
                                  scoring='neg_mean_absolute_error',
                                  groups=g, n_jobs=-1)
    scores_r2  = cross_val_score(gb_fresh, X_train_sc, y_train, cv=cv,
                                  scoring='r2', groups=g, n_jobs=-1)
    cv_strategy_results.append({
        'Strategy':    name,
        'CV_MAE_mean': round(-scores_mae.mean(), 4),
        'CV_MAE_std':  round(scores_mae.std(), 4),
        'CV_R2_mean':  round(scores_r2.mean(), 4),
        'CV_R2_std':   round(scores_r2.std(), 4),
        'Folds':       len(scores_mae),
    })
cv_strategy_df = pd.DataFrame(cv_strategy_results)
print("CV Strategy Comparison:")
print(cv_strategy_df.to_string(index=False))

In [ ]:
# ── Learning curve ───────────────────────────────────────────────────────
fig = learning_curve_plot(gb_fresh, X_train_sc, y_train,
      model_name=best_name, cv=5,
      save_path=paths.fig('nb06_learning_curve.png'))
plt.show()
print('INTERPRETATION: If the training and validation curves converge as training size increases,')
print('the model has reached a good bias-variance balance. A persistent gap indicates high variance')
print('(overfitting), while curves that both plateau at high error indicate high bias (underfitting).')
print('Given the small dataset, some variance is expected and is managed by depth constraints')
print('and the minimum samples per leaf parameter.')

In [ ]:
# ── Prediction errors by year ─────────────────────────────────────────────
# Evaluate across all years (train + test)
X_all_sc = scaler.transform(model_df[model_feats].values)
y_all     = model_df[TARGET].values
y_all_pred = best_model.predict(X_all_sc)

df_errors = model_df[['year','region',TARGET]].copy()
df_errors['predicted'] = y_all_pred
df_errors['error']     = df_errors[TARGET] - df_errors['predicted']
df_errors['abs_error'] = df_errors['error'].abs()

print("Mean absolute error by year:")
print(df_errors.groupby('year')['abs_error'].agg(['mean','std','max']).round(4))

In [ ]:
# ── Top and bottom prediction errors ─────────────────────────────────────
print("\n10 largest prediction errors:")
print(df_errors.nlargest(10, 'abs_error')[['year','region',TARGET,'predicted','error']].to_string(index=False))

print("\n10 most accurate predictions:")
print(df_errors.nsmallest(10, 'abs_error')[['year','region',TARGET,'predicted','error']].to_string(index=False))

In [ ]:
# ── Save evaluation outputs ──────────────────────────────────────────────
save_dataframe(df_errors, paths.table('prediction_errors.csv'))
save_dataframe(cv_strategy_df, paths.table('cv_strategy_comparison.csv'))

print("\nEvaluation complete. Outputs saved.")
print(f"  Test R²: {metrics['R2']}")
print(f"  Test MAE: {metrics['MAE']}%")